# Autonomous Overhead Crane: Object Detection

This notebook documents the project's two YOLO detectors and provides a reproducible training and evaluation workflow for the truck-and-cargo detector.

- The truck-and-cargo model detects `mytruck` and `fardeaux`.
- The separate cone model detects `cone`; its trained checkpoint is included, but its training dataset is not included in this repository.

Run the cells in order. Training is opt-in; by default the notebook evaluates the included truck-and-cargo checkpoint on the held-out test split.

## 1. Install dependencies

Run this cell if the packages are not already installed in the active notebook environment.

In [ ]:
%pip install ultralytics pyyaml

## 2. Configure project paths and model classes

The dataset export is Pascal VOC XML. The conversion cell below prepares a temporary YOLO-format copy and keeps the original export unchanged. The dataset may be placed in `data/truck-cargo-v4` or in a sibling `Dataset` folder.

In [ ]:
from pathlib import Path
import shutil
import tempfile
import xml.etree.ElementTree as ET

import yaml
from IPython.display import Image, display
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models").is_dir() and (PROJECT_ROOT / "Detection").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "Detection"

DATASET_CANDIDATES = [
    PROJECT_ROOT / "data" / "truck-cargo-v4",
    PROJECT_ROOT.parent / "Dataset",
]
DATASET_SOURCE = next(
    (path for path in DATASET_CANDIDATES if all((path / split).is_dir() for split in ("train", "valid", "test"))),
    None,
)
if DATASET_SOURCE is None:
    raise FileNotFoundError(
        "Dataset not found. Place the Roboflow Pascal VOC export in "
        "data/truck-cargo-v4/{train,valid,test}."
    )

MODELS_DIR = PROJECT_ROOT / "models"
TRUCK_CARGO_WEIGHTS = MODELS_DIR / "truck-cargo" / "best.pt"
CONE_WEIGHTS = MODELS_DIR / "cone" / "best.pt"
RUNS_DIR = PROJECT_ROOT / "runs"
CLASS_NAMES = ["fardeaux", "mytruck"]
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}

# Set True to train a new truck-and-cargo detector before evaluation.
TRAIN_TRUCK_CARGO = False

print(f"Project: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_SOURCE}")
print(f"Truck/cargo classes: {CLASS_NAMES}")
print(f"Cone checkpoint available: {CONE_WEIGHTS.is_file()}")

## 3. Convert Pascal VOC annotations to YOLO format

The Roboflow version 4 export uses XML annotations. This cell converts each bounding box to normalized YOLO coordinates in a temporary directory and creates a dataset YAML file for Ultralytics.

In [ ]:
dataset_temp = tempfile.TemporaryDirectory(prefix="truck_cargo_yolo_")
YOLO_DATASET = Path(dataset_temp.name)

for split in ("train", "valid", "test"):
    source_dir = DATASET_SOURCE / split
    image_dir = YOLO_DATASET / "images" / split
    label_dir = YOLO_DATASET / "labels" / split
    image_dir.mkdir(parents=True)
    label_dir.mkdir(parents=True)

    image_paths = sorted(source_dir.glob("*.jpg"))
    if not image_paths:
        raise FileNotFoundError(f"No JPG images found in {source_dir}")

    for image_path in image_paths:
        annotation_path = image_path.with_suffix(".xml")
        if not annotation_path.is_file():
            raise FileNotFoundError(f"Missing annotation: {annotation_path}")

        annotation = ET.parse(annotation_path).getroot()
        size = annotation.find("size")
        image_width = int(size.findtext("width"))
        image_height = int(size.findtext("height"))
        label_lines = []

        for obj in annotation.findall("object"):
            class_name = obj.findtext("name", "").strip().lower()
            if class_name not in CLASS_TO_ID:
                raise ValueError(f"Unexpected class {class_name!r} in {annotation_path}")

            box = obj.find("bndbox")
            xmin = float(box.findtext("xmin"))
            ymin = float(box.findtext("ymin"))
            xmax = float(box.findtext("xmax"))
            ymax = float(box.findtext("ymax"))
            x_center = (xmin + xmax) / (2 * image_width)
            y_center = (ymin + ymax) / (2 * image_height)
            box_width = (xmax - xmin) / image_width
            box_height = (ymax - ymin) / image_height

            if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < box_width <= 1 and 0 < box_height <= 1):
                raise ValueError(f"Invalid bounding box in {annotation_path}")

            label_lines.append(
                f"{CLASS_TO_ID[class_name]} {x_center:.8f} {y_center:.8f} "
                f"{box_width:.8f} {box_height:.8f}"
            )

        shutil.copy2(image_path, image_dir / image_path.name)
        (label_dir / f"{image_path.stem}.txt").write_text("\n".join(label_lines), encoding="utf-8")

DATA_YAML = YOLO_DATASET / "data.yaml"
DATA_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(YOLO_DATASET),
            "train": "images/train",
            "val": "images/valid",
            "test": "images/test",
            "names": CLASS_NAMES,
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)

print(f"Converted YOLO dataset: {YOLO_DATASET}")

## 4. Train or load the truck-and-cargo detector

Set `TRAIN_TRUCK_CARGO = True` in the configuration cell to train a new YOLOv8n model for 50 epochs at 416-pixel image size. Otherwise, the included checkpoint is loaded for evaluation. This detector has two classes: `fardeaux` and `mytruck`.

In [ ]:
if TRAIN_TRUCK_CARGO:
    model = YOLO("yolov8n.pt")
    model.train(
        data=str(DATA_YAML),
        epochs=50,
        imgsz=416,
        batch=8,
        project=str(RUNS_DIR),
        name="truck-cargo-v4",
        exist_ok=True,
    )
    TRUCK_CARGO_WEIGHTS = RUNS_DIR / "truck-cargo-v4" / "weights" / "best.pt"
    if not TRUCK_CARGO_WEIGHTS.is_file():
        raise FileNotFoundError(f"Training did not produce best.pt: {TRUCK_CARGO_WEIGHTS}")
else:
    if not TRUCK_CARGO_WEIGHTS.is_file():
        raise FileNotFoundError(f"Truck/cargo checkpoint not found: {TRUCK_CARGO_WEIGHTS}")
    print(f"Using existing checkpoint: {TRUCK_CARGO_WEIGHTS}")

truck_cargo_model = YOLO(str(TRUCK_CARGO_WEIGHTS))
print(f"Model classes: {truck_cargo_model.names}")

## 5. Evaluate on the held-out test split

The test split is used only for evaluation. Ultralytics reports precision, recall, mAP@50, and mAP@50-95, and saves diagnostic plots.

In [ ]:
metrics = truck_cargo_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=416,
    batch=8,
    plots=True,
)

print("Truck-and-cargo test metrics")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

confusion_matrix_path = Path(metrics.save_dir) / "confusion_matrix.png"
if not confusion_matrix_path.is_file():
    raise FileNotFoundError(f"Confusion matrix was not saved: {confusion_matrix_path}")
display(Image(filename=str(confusion_matrix_path)))

## 6. Load the separate cone detector

The cone checkpoint is provided as a separate single-class model. It is not merged into the truck-and-cargo detector. The cone training images and annotations were not present in the supplied project, so this notebook can load and use the checkpoint but cannot reproduce its training. Set `CONE_IMAGE` to an image path to run an example prediction.

In [ ]:
if not CONE_WEIGHTS.is_file():
    raise FileNotFoundError(f"Cone checkpoint not found: {CONE_WEIGHTS}")

cone_model = YOLO(str(CONE_WEIGHTS))
print(f"Cone model classes: {cone_model.names}")

CONE_IMAGE = None  # Example: PROJECT_ROOT / "data" / "example_cone.jpg"
if CONE_IMAGE is not None:
    cone_results = cone_model.predict(source=str(CONE_IMAGE), save=True)
    display(Image(filename=str(Path(cone_results[0].save_dir) / Path(CONE_IMAGE).name)))